# OLS vs Ridge sanity check for PSR fitting

Verifies that plain OLS (np.linalg.lstsq) and Ridge (lam=1e-4) produce materially equivalent PSR weights, justifying the choice of OLS for the final pipeline.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
# Honest verification: do OLS (np.linalg.lstsq) and Ridge (lam=1e-4) produce
# materially different PSR weights?
# If the script can't find data files, it tells you exactly which files are
# missing so you can fix the path or the registry.

In [ ]:
import os
import glob
import numpy as np
import h5py

ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
RATE = 125
SUBSAMPLE = 4
MIN_SAMP = 200

TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}

# Healthy-only registry (only what we need for PSR weight fitting)
REGISTRY_HEALTHY = {
    "T1_healthy": ("T1_PickPlace/Healthy", "UR5_T1_healthy_180cyc_*.h5", "T1"),
    "T2_healthy": ("T2_Assembly/Healthy",  "UR5_T2_healthy_180cyc_*.h5", "T2"),
    "T3_healthy": ("T3_Palletize/Healthy", "UR5_T3_healthy_183cyc_*.h5", "T3"),
}

# Use T1 LOTO fold (train on T2+T3, like Table 2 row 1)
HELD_OUT = "T1"

print(f"ROOT: {ROOT}")
print(f"BASE: {BASE}")
print(f"Held-out fold: {HELD_OUT}")
print(f"Training tasks: {[t for t in ['T1','T2','T3'] if t != HELD_OUT]}")
print()

In [ ]:
UR5_DH = {"a":     [0, -0.42500, -0.39225, 0, 0, 0],
          "d":     [0.089159, 0, 0, 0.10915, 0.09465, 0.0823],
          "alpha": [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]}
UR5_MASS = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM = [[0.0,    -0.02561, 0.00193],
           [0.21250, 0.0,     0.11336],
           [0.11993, 0.0,     0.02650],
           [0.0,    -0.00180, 0.01634],
           [0.0,     0.00180, 0.01634],
           [0.0,     0.0,    -0.00116]]
GRAVITY = np.array([0, 0, -9.81])

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([[ct, -st*ca,  st*sa, a*ct],
                     [st,  ct*ca, -ct*sa, a*st],
                     [0,   sa,     ca,    d   ],
                     [0,   0,      0,     1   ]])

def gravity_torque(q, payload_mass=0.0, payload_com=np.array([0, 0, 0.05])):
    a, d, alpha = UR5_DH["a"], UR5_DH["d"], UR5_DH["alpha"]
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(a[i], d[i], alpha[i], q[i]))
    com_positions = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3] for i in range(6)]
    if payload_mass > 0:
        com_positions.append((T[6] @ np.array([*payload_com, 1.0]))[:3])
        masses = UR5_MASS + [payload_mass]
    else:
        masses = UR5_MASS
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        q_plus = q.copy(); q_plus[i] += dq
        T_plus = [np.eye(4)]
        for j in range(6):
            T_plus.append(T_plus[-1] @ dh_transform(a[j], d[j], alpha[j], q_plus[j]))
        for j in range(len(masses)):
            if j < 6:
                com_plus = (T_plus[j+1] @ np.array([*UR5_COM[j], 1.0]))[:3]
            else:
                com_plus = (T_plus[6] @ np.array([*payload_com, 1.0]))[:3]
            dp_dq = (com_plus - com_positions[j]) / dq
            tau_g[i] += masses[j] * GRAVITY @ dp_dq
    return tau_g

In [ ]:
print("Searching for healthy data files...")
print()

resolved_files = {}
for tag, (subdir, pattern, task) in REGISTRY_HEALTHY.items():
    if task == HELD_OUT:
        print(f"  {tag} → held out, skipped")
        continue
    full_pattern = os.path.join(BASE, subdir, pattern)
    hits = glob.glob(full_pattern)
    if hits:
        resolved_files[tag] = (hits[0], task)
        print(f"  {tag} → found: {os.path.basename(hits[0])}")
    else:
        print(f"  {tag} → NOT FOUND")
        print(f"             pattern: {full_pattern}")

if not resolved_files:
    print()
    print("ERROR: No healthy data files were found. Please check that:")
    print(f"  1. The BASE path exists: {BASE}")
    print(f"  2. It contains subfolders T1_PickPlace/Healthy, T2_Assembly/Healthy, T3_Palletize/Healthy")
    print(f"  3. The healthy .h5 files are present")
    print()
    print("Diagnostic — listing what is actually in BASE:")
    if os.path.isdir(BASE):
        for entry in os.listdir(BASE):
            print(f"    {entry}")
    else:
        print(f"    BASE directory does not exist: {BASE}")
    raise SystemExit(1)

print()

In [ ]:
def load_cycles(filepath):
    with h5py.File(filepath, "r") as f:
        q       = f["actual_q"][:]
        qd      = f["actual_qd"][:]
        current = f["actual_current"][:]
        cn      = f["cycle_number"][:].ravel()
    out = []
    for c in np.unique(cn[cn > 0]):
        m = cn == c
        if m.sum() >= MIN_SAMP:
            out.append({"q": q[m], "qd": qd[m], "current": current[m]})
    return out

all_healthy = []
for tag, (filepath, task) in resolved_files.items():
    cycles = load_cycles(filepath)
    if len(cycles) > 2:
        cycles = cycles[1:-1]   # trim warm-up and cool-down, matches NB8
    for c in cycles:
        c["task"] = task
    all_healthy.extend(cycles)
    print(f"  {task}: loaded {len(cycles)} healthy cycles from {os.path.basename(filepath)}")

print()
print(f"Total training cycles: {len(all_healthy)}")
print()

if len(all_healthy) == 0:
    raise SystemExit("No cycles loaded — check MIN_SAMP and HDF5 structure.")

In [ ]:
print("Building per-joint regressor matrices...")

train_Phi = {j: [] for j in range(6)}
train_I   = {j: [] for j in range(6)}

for c in all_healthy:
    payload = TASK_PAYLOAD[c["task"]]
    q_sub   = c["q"][::SUBSAMPLE]
    qd_full = c["qd"]
    cur_sub = c["current"][::SUBSAMPLE]
    for t in range(len(q_sub)):
        tau_g  = gravity_torque(q_sub[t], payload_mass=payload)
        t_full = t * SUBSAMPLE
        for j in range(6):
            if 0 < t_full < len(qd_full) - 1:
                qdd_j = (qd_full[t_full+1, j] - qd_full[t_full-1, j]) * RATE / 2
            else:
                qdd_j = 0.0
            phi = np.array([tau_g[j], qd_full[t_full, j],
                            np.sign(qd_full[t_full, j]), qdd_j, 1.0])
            train_Phi[j].append(phi)
            i_val = cur_sub[t, j] if t < len(cur_sub) else c["current"][t_full, j]
            train_I[j].append(i_val)

for j in range(6):
    print(f"  J{j}: {len(train_Phi[j])} timesteps")
print()

# Verify shapes BEFORE running lstsq (this is what caused the v1 crash)
for j in range(6):
    n = len(train_Phi[j])
    if n == 0:
        raise SystemExit(f"J{j}: no timesteps collected. Check cycle loading.")

In [ ]:
print(f"{'='*88}")
print(f"OLS vs Ridge (lam=1e-4) — per-joint weight comparison")
print(f"Held-out fold: {HELD_OUT}   Training: {[t for t in TASK_PAYLOAD.keys() if t != HELD_OUT and t != 'T4']}")
print(f"{'='*88}")
print()
print(f"{'Joint':<6} {'Method':<8} {'w0 (grav)':>11} {'w1 (visc)':>11} "
      f"{'w2 (Coul)':>11} {'w3 (iner)':>11} {'w4 (bias)':>11} {'RMSE (A)':>10}")
print("-" * 88)

max_rel_diff = 0.0
weights_ols = {}
weights_ridge = {}

for j in range(6):
    Phi_j = np.array(train_Phi[j])
    I_j   = np.array(train_I[j])
    
    # Sanity check
    assert Phi_j.ndim == 2 and Phi_j.shape[1] == 5, f"J{j}: Phi shape {Phi_j.shape}"
    assert I_j.ndim == 1 and I_j.shape[0] == Phi_j.shape[0], f"J{j}: I shape {I_j.shape}"

    # OLS
    w_ols, *_ = np.linalg.lstsq(Phi_j, I_j, rcond=None)
    rmse_ols  = np.sqrt(np.mean((I_j - Phi_j @ w_ols) ** 2))

    # Ridge with lam = 1e-4
    lam = 1e-4
    A = Phi_j.T @ Phi_j + lam * np.eye(5)
    b = Phi_j.T @ I_j
    w_ridge = np.linalg.solve(A, b)
    rmse_ridge = np.sqrt(np.mean((I_j - Phi_j @ w_ridge) ** 2))

    weights_ols[j]   = w_ols
    weights_ridge[j] = w_ridge

    # Per-coefficient relative difference
    rel_diffs = np.abs(w_ridge - w_ols) / (np.abs(w_ols) + 1e-12)
    max_rel_diff = max(max_rel_diff, rel_diffs.max())

    print(f"J{j:<5} {'OLS':<8} "
          f"{w_ols[0]:>11.6f} {w_ols[1]:>11.6f} {w_ols[2]:>11.6f} "
          f"{w_ols[3]:>11.6f} {w_ols[4]:>11.6f} {rmse_ols:>10.6f}")
    print(f"      {'Ridge':<8} "
          f"{w_ridge[0]:>11.6f} {w_ridge[1]:>11.6f} {w_ridge[2]:>11.6f} "
          f"{w_ridge[3]:>11.6f} {w_ridge[4]:>11.6f} {rmse_ridge:>10.6f}")
    print(f"      {'rel diff':<8} "
          f"{rel_diffs[0]:>11.2e} {rel_diffs[1]:>11.2e} {rel_diffs[2]:>11.2e} "
          f"{rel_diffs[3]:>11.2e} {rel_diffs[4]:>11.2e}")
    print()

In [ ]:
print(f"\n{'='*88}")
print(f"VERDICT")
print(f"{'='*88}")
print(f"\nMax relative coefficient difference (any joint, any term): {max_rel_diff:.6e}")
print()

if max_rel_diff < 1e-3:
    print("CONCLUSION: OLS and Ridge produce essentially identical weights")
    print("            (max relative difference < 0.1%).")
    print()
    print("            The 'methodological inconsistency' between NB8/NB10/NB10d (OLS)")
    print("            and NB10c/NB10cd (Ridge lam=1e-4) is numerical noise only.")
    print()
    print("            The manuscript can honestly describe PSR weight fitting as")
    print("            OLS for ALL detectors. The GMM row in Table 3 is consistent.")
elif max_rel_diff < 1e-1:
    print("CONCLUSION: Weights differ by less than 10%.")
    print("            Probably produces similar AUROC but worth double-checking.")
    print("            Re-run the GMM detector with OLS weights and compare to Table 3's GMM row.")
else:
    print("CONCLUSION: Weights differ materially (>10%).")
    print("            The GMM row in Table 3 is NOT consistent with PSR Z-Score/OC-SVM/IsoForest")
    print("            because they use different fits.")
    print("            Recommendation: re-run GMM with OLS weights, OR disclose Ridge in methods.")

print()